# 05 — Threshold Tuning and Final Evaluation

This notebook selects the fraud decision threshold using a temporal validation
set, retrains the final Random Forest model on the full training period, and
evaluates it once on the reserved temporal holdout test set.

The final test set is not used during model or threshold selection.

In [1]:
from pathlib import Path
import json
import joblib

import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

DATA_DIR = Path("../data")
ARTIFACT_DIR = Path("../artifacts")

ARTIFACT_DIR.mkdir(exist_ok=True)

In [2]:
df = pd.read_csv(
    DATA_DIR / "transaction_features_temporal_v1.csv"
)

print("Shape:", df.shape)
print("Missing:", df.isna().sum().sum())
print("Duplicate tx_id:", df["tx_id"].duplicated().sum())

Shape: (200000, 19)
Missing: 0
Duplicate tx_id: 0


## 1. Recreate the Temporal Development Split

The same temporal boundaries used during model development are reproduced here.

The final holdout begins after the 80th percentile of `time_step`.
The earlier period is split again to obtain the validation set used for
threshold selection.

In [3]:
test_cutoff = df["time_step"].quantile(0.80)

train_period = df[
    df["time_step"] <= test_cutoff
].copy()

test_df = df[
    df["time_step"] > test_cutoff
].copy()

validation_cutoff = train_period[
    "time_step"
].quantile(0.80)

train_core_df = train_period[
    train_period["time_step"] <= validation_cutoff
].copy()

val_df = train_period[
    train_period["time_step"] > validation_cutoff
].copy()

print("Test cutoff:", test_cutoff)
print("Validation cutoff:", validation_cutoff)

print("Train core:", train_core_df.shape)
print("Validation:", val_df.shape)
print("Final test:", test_df.shape)

Test cutoff: 356.0
Validation cutoff: 302.0
Train core: (128659, 19)
Validation: (31605, 19)
Final test: (39736, 19)


In [4]:
temporal_graph_features = [
    "tx_type",
    "amount",
    "log_amount",
    "dst_balance_before",

    "receiver_prior_tx_count",
    "receiver_prior_total_amount",
    "receiver_prior_avg_amount",

    "receiver_prior_7d_tx_count",
    "receiver_prior_7d_total_amount"
]

In [5]:
def build_rf_model():
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                ["tx_type"]
            )
        ],
        remainder="passthrough"
    )

    return Pipeline([
        ("preprocessor", preprocessor),

        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            )
        )
    ])

## 2. Threshold Selection on Validation Data

The model is first trained only on the train-core period.

Decision thresholds are evaluated exclusively on the later validation period.
The final holdout test set remains untouched.

In [6]:
X_train_core = train_core_df[
    temporal_graph_features
].copy()

y_train_core = train_core_df[
    "label_fraud"
].copy()

X_val = val_df[
    temporal_graph_features
].copy()

y_val = val_df[
    "label_fraud"
].copy()

rf_tuning_model = build_rf_model()

rf_tuning_model.fit(
    X_train_core,
    y_train_core
)

val_prob = rf_tuning_model.predict_proba(
    X_val
)[:, 1]

In [7]:
threshold_results = []

for threshold in np.arange(
    0.05,
    1.00,
    0.05
):
    val_pred = (
        val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        val_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        val_pred
    ).ravel()

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fraud_caught_TP": tp,
        "fraud_missed_FN": fn,
        "false_alarms_FP": fp
    })

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df.round(4)

,threshold,precision,recall,f1,fraud_caught_TP,fraud_missed_FN,false_alarms_FP
0,0.05,0.1649,0.6205,0.2606,345,211,1747
1,0.10,0.2382,0.5809,0.3379,323,233,1033
2,0.15,0.2856,0.5522,0.3765,307,249,768
3,0.20,0.3123,0.5432,0.3966,302,254,665
4,0.25,0.3487,0.5180,0.4168,288,268,538
5,0.30,0.3794,0.4838,0.4253,269,287,440
6,0.35,0.3941,0.4586,0.4239,255,301,392
7,0.40,0.3977,0.4371,0.4165,243,313,368
8,0.45,0.3977,0.4335,0.4148,241,315,365
9,0.50,0.3980,0.4281,0.4125,238,318,360


In [9]:
best_coarse = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

best_coarse

threshold            0.300000
precision            0.379408
recall               0.483813
f1                   0.425296
fraud_caught_TP    269.000000
fraud_missed_FN    287.000000
false_alarms_FP    440.000000
Name: 5, dtype: float64

In [10]:
fine_threshold_results = []

for threshold in np.arange(
    0.25,
    0.36,
    0.01
):
    val_pred = (
        val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        val_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        val_pred
    ).ravel()

    fine_threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fraud_caught_TP": tp,
        "fraud_missed_FN": fn,
        "false_alarms_FP": fp
    })

fine_threshold_df = pd.DataFrame(
    fine_threshold_results
)

fine_threshold_df.round(4)

,threshold,precision,recall,f1,fraud_caught_TP,fraud_missed_FN,false_alarms_FP
0,0.25,0.3487,0.5180,0.4168,288,268,538
1,0.26,0.3547,0.5072,0.4175,282,274,513
2,0.27,0.3578,0.5000,0.4171,278,278,499
3,0.28,0.3636,0.4892,0.4172,272,284,476
4,0.29,0.3776,0.4856,0.4249,270,286,445
5,0.30,0.3801,0.4820,0.4251,268,288,437
6,0.31,0.3878,0.4784,0.4283,266,290,420
7,0.32,0.3873,0.4730,0.4259,263,293,416
8,0.33,0.3862,0.4640,0.4216,258,298,410
9,0.34,0.3879,0.4604,0.4211,256,300,404


In [11]:
best_fine = fine_threshold_df.loc[
    fine_threshold_df["f1"].idxmax()
]

best_fine

threshold            0.310000
precision            0.387755
recall               0.478417
f1                   0.428341
fraud_caught_TP    266.000000
fraud_missed_FN    290.000000
false_alarms_FP    420.000000
Name: 6, dtype: float64

In [12]:
selected_threshold = float(
    best_fine["threshold"]
)

print(
    "Selected threshold:",
    selected_threshold
)

Selected threshold: 0.31


## 3. Retrain Final Model

After threshold selection, the final model is retrained on the complete
pre-holdout training period.

No threshold or model selection is performed using the final test period.

In [13]:
X_train_final = train_period[
    temporal_graph_features
].copy()

y_train_final = train_period[
    "label_fraud"
].copy()

X_test = test_df[
    temporal_graph_features
].copy()

y_test = test_df[
    "label_fraud"
].copy()

final_rf_model = build_rf_model()

final_rf_model.fit(
    X_train_final,
    y_train_final
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
test_prob = final_rf_model.predict_proba(
    X_test
)[:, 1]

test_pred = (
    test_prob >= selected_threshold
).astype(int)

In [16]:
final_precision = precision_score(
    y_test,
    test_pred
)

final_recall = recall_score(
    y_test,
    test_pred
)

final_f1 = f1_score(
    y_test,
    test_pred
)

final_accuracy = accuracy_score(
    y_test,
    test_pred
)

final_cm = confusion_matrix(
    y_test,
    test_pred
)

print("Selected threshold:", selected_threshold)

print("\nConfusion Matrix:")
print(final_cm)

print(
    "\nFraud precision:",
    round(final_precision, 4)
)

print(
    "Fraud recall:",
    round(final_recall, 4)
)

print(
    "Fraud F1:",
    round(final_f1, 4)
)

print(
    "Accuracy:",
    round(final_accuracy, 4)
)

Selected threshold: 0.31

Confusion Matrix:
[[35029   463]
 [ 2221  2023]]

Fraud precision: 0.8138
Fraud recall: 0.4767
Fraud F1: 0.6012
Accuracy: 0.9325


In [17]:
default_pred = (
    test_prob >= 0.50
).astype(int)

default_cm = confusion_matrix(
    y_test,
    default_pred
)

tn_050, fp_050, fn_050, tp_050 = (
    default_cm.ravel()
)

tn_final, fp_final, fn_final, tp_final = (
    final_cm.ravel()
)

print(
    "Additional fraud caught:",
    tp_final - tp_050
)

print(
    "Additional false alarms:",
    fp_final - fp_050
)

Additional fraud caught: 182
Additional false alarms: 89


### Threshold trade-off

Lowering the decision threshold increases fraud recall at the cost of additional
false alarms.

The selected threshold is not chosen on the final test set; it is fixed in
advance using validation F1.

In [18]:
final_metrics = pd.DataFrame([
    {
        "threshold": selected_threshold,
        "precision": final_precision,
        "recall": final_recall,
        "f1": final_f1,
        "accuracy": final_accuracy,
        "true_negatives": final_cm[0, 0],
        "false_positives": final_cm[0, 1],
        "false_negatives": final_cm[1, 0],
        "true_positives": final_cm[1, 1]
    }
])

final_metrics.to_csv(
    ARTIFACT_DIR / "final_metrics.csv",
    index=False
)

final_metrics.round(4)

,threshold,precision,recall,f1,accuracy,true_negatives,false_positives,false_negatives,true_positives
0,0.31,0.8138,0.4767,0.6012,0.9325,35029,463,2221,2023


In [19]:
final_predictions = test_df[[
    "tx_id",
    "time_step",
    "label_fraud"
]].copy()

final_predictions["fraud_probability"] = (
    test_prob
)

final_predictions["predicted_fraud"] = (
    test_pred
)

final_predictions.to_csv(
    ARTIFACT_DIR
    / "final_test_predictions.csv",
    index=False
)

In [21]:
joblib.dump(
    final_rf_model,
    ARTIFACT_DIR
    / "fraud_rf_temporal_graph_v1.joblib"
)

['..\\artifacts\\fraud_rf_temporal_graph_v1.joblib']

In [22]:
model_config = {
    "model": "RandomForestClassifier",
    "version": "v1",

    "selected_threshold": (
        selected_threshold
    ),

    "threshold_metric": (
        "validation_f1"
    ),

    "test_cutoff": float(
        test_cutoff
    ),

    "validation_cutoff": float(
        validation_cutoff
    ),

    "features": (
        temporal_graph_features
    ),

    "model_parameters": {
        "n_estimators": 200,
        "class_weight": (
            "balanced_subsample"
        ),
        "random_state": 42
    },

    "notes": (
        "Sender-balance-derived features "
        "excluded after ablation."
    )
}

with open(
    ARTIFACT_DIR / "model_config.json",
    "w"
) as f:
    json.dump(
        model_config,
        f,
        indent=4
    )

In [23]:
for path in ARTIFACT_DIR.iterdir():
    print(path.name)

final_metrics.csv
final_test_predictions.csv
fraud_rf_temporal_graph_v1.joblib
model_config.json
